# 🧠 Notebook 03 — Prompt Engineering Showcase

This notebook demonstrates the prompt engineering techniques used in LectureForge AI.

It's designed to show **how prompt design directly affects output quality** — 
the difference between a vague prompt and a well-engineered one is enormous.

Each section shows:
1. A naive/bad prompt and its output
2. The engineered prompt and its output
3. Analysis of what changed and why it matters

---
**Relevance to educational AI:**
This is exactly the skill needed to build AI-enhanced learning systems —
crafting prompts that produce pedagogically sound, consistent, LMS-ready content.

In [ ]:
import sys, os
sys.path.append('..')
from dotenv import load_dotenv
load_dotenv('../.env')

from openai import OpenAI
client = OpenAI()

SAMPLE_CONTEXT = """
Machine learning is a subset of artificial intelligence that enables computers to learn
from data without being explicitly programmed. There are three main types: supervised
learning (labelled data), unsupervised learning (unlabelled data), and reinforcement
learning (reward-based). Key algorithms include linear regression, decision trees,
neural networks, and k-means clustering. Overfitting occurs when a model performs well
on training data but poorly on new data. Cross-validation is used to evaluate model
performance. Feature engineering involves selecting and transforming input variables
to improve model accuracy.
"""

def ask(prompt, temperature=0.3):
    r = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=600, temperature=temperature
    )
    return r.choices[0].message.content.strip()

print("✅ Setup complete")

---
## Experiment 1 — Learning Objectives
### Bad prompt vs Engineered prompt

In [ ]:
# ❌ Bad prompt — vague, no constraints, no format
bad = ask("Write some learning objectives for a machine learning lecture.")
print("❌ BAD PROMPT OUTPUT:")
print(bad)

In [ ]:
# ✅ Engineered prompt — role, constraints, Bloom's, JSON output, grounded in content
engineered = ask(f"""
You are an expert instructional designer with deep knowledge of Bloom's Taxonomy.

Based ONLY on the lecture content below, generate exactly 6 learning objectives.

Each objective must:
- Start with a measurable action verb from Bloom's Taxonomy
- Be specific to content actually covered in this lecture
- Be achievable by someone who completes this single module
- Be written from the learner's perspective

Bloom's levels to use (mix across objectives): Remember, Understand, Apply, Analyse, Evaluate, Create

Return ONLY valid JSON:
[{{"objective": "...", "bloom_level": "Understand", "verb": "Explain"}}]

LECTURE CONTENT:
{SAMPLE_CONTEXT}
""")

print("✅ ENGINEERED PROMPT OUTPUT:")
print(engineered)

### 🔍 Analysis — What changed?

| Factor | Bad prompt | Engineered prompt |
|---|---|---|
| **Role** | None | Expert instructional designer |
| **Count** | Unspecified | Exactly 6 |
| **Grounding** | No — from model memory | Yes — from lecture content only |
| **Format** | Free text | JSON with defined schema |
| **Pedagogy** | None | Bloom's Taxonomy alignment |
| **Measurability** | Vague | Action verbs required |

Result: The bad prompt produces generic, unmeasurable objectives.
The engineered prompt produces LMS-ready, pedagogically sound outcomes.

---
## Experiment 2 — Temperature Effect
Same prompt, different temperature — shows how temperature affects consistency

In [ ]:
prompt = f"In 2 sentences, summarise the most important concept from this lecture:\n{SAMPLE_CONTEXT}"

for temp in [0.0, 0.3, 0.7, 1.0]:
    result = ask(prompt, temperature=temp)
    print(f"Temperature {temp}:")
    print(result)
    print()

### 🔍 Analysis — Temperature

| Temperature | Output quality for educational content |
|---|---|
| **0.0** | Deterministic — identical every run. Good for extraction, too rigid for full content |
| **0.3** | ✅ Our choice — consistent tone and structure, slight natural variation |
| **0.7** | More varied phrasing — less predictable for LMS scale |
| **1.0** | Creative but inconsistent — unsuitable for structured educational content |

---
## Experiment 3 — JSON Schema Enforcement
Shows how explicit schema in the prompt prevents parsing failures

In [ ]:
import json

# Without schema — unpredictable key names
no_schema = ask(f"Create 3 flashcards from this content as JSON:\n{SAMPLE_CONTEXT}")
print("Without schema:", no_schema[:300])
print()

# With schema — always parseable
with_schema = ask(f"""
Create exactly 3 flashcards from this content.
Return ONLY valid JSON, no markdown backticks, no preamble:
[
  {{"term": "term name here", "definition": "clear definition here"}}
]

CONTENT:
{SAMPLE_CONTEXT}
""")

print("With schema:", with_schema)

# Try parsing — this should succeed reliably
try:
    cards = json.loads(with_schema)
    print(f"\n✅ Parsed successfully — {len(cards)} flashcards")
    for c in cards:
        print(f"  {c['term']}: {c['definition'][:60]}...")
except json.JSONDecodeError as e:
    print(f"❌ Parse failed: {e}")

---
## Summary — Prompt Engineering Principles Used

| Principle | Why it matters |
|---|---|
| **Role priming** | Sets the model's perspective and domain expertise |
| **Hard constraints** | Exact counts and format requirements eliminate variance |
| **Bloom's Taxonomy** | Produces pedagogically sound learning objectives |
| **JSON schema** | Guarantees parseable, programmatically usable outputs |
| **RAG grounding** | Prevents hallucination — output reflects actual content |
| **Low temperature** | Ensures consistency across hundreds of module generations |

These principles are what separate a toy LLM app from a production-ready educational AI system.